Corner plot of a nautilus run, read straight from its `.h5` file. It works while the run is still going: rerun the last cell to refresh. Until the run leaves the exploration phase (`explored = False`), the posterior is rough.

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'   # read the file while the sampler writes it
import numpy as np
import matplotlib.pyplot as plt
import corner
from nautilus import Sampler
import beyond21.log_prob as NF
import beyond21.run_table as run_table

chain_num = '0'
results_dir = 'Beyond21Naut/chains'

In [ ]:
# Rebuild the parameter names from the settings the run recorded
row = next(r for r in run_table.read(f"{results_dir}/{run_table.FILENAME}")
           if r['sampler'] == 'nautilus' and r['chain_num'] == chain_num)
b = lambda k: row[k] == 'True'
names = list(NF.get_par_priors(row['run_mode'], saras=b('saras'), sfe_21cmFAST=b('sfe_21cmFAST'),
                               fixed=NF.parse_fixed(row['fixed']), popII_only=b('popII_only')))
print(run_table.show(f"{results_dir}/{run_table.FILENAME}"))
print(names)

In [ ]:
# nautilus stores unit-cube points and maps them through the prior on output,
# so pass the run's real prior transform. The likelihood is a dummy: we never sample.
prior = lambda u: NF.prior_transform(u, row['run_mode'], saras=b('saras'),
                                     sfe_21cmFAST=b('sfe_21cmFAST'),
                                     fixed=NF.parse_fixed(row['fixed']),
                                     popII_only=b('popII_only'))
s = Sampler(prior, lambda x: 0., n_dim=len(names), n_live=1000,
            filepath=f"{results_dir}/nautilus{chain_num}.h5", resume=True)
pts, log_w, log_l = s.posterior()
print(f"{len(pts)} points, n_eff = {s.n_eff:.0f}, explored = {s.explored}")

LATEX = {
    't_star': r'$t_\star$',
    'log10FstarIII': r'$\log_{10} f_{\star,\mathrm{III}}$', 'alpha_starIII': r'$\alpha_{\star,\mathrm{III}}$',
    'log10McutIII': r'$\log_{10} M_{\mathrm{cut,III}}$',
    'log10FstarII': r'$\log_{10} f_{\star,\mathrm{II}}$', 'log10Mpivot': r'$\log_{10} M_\mathrm{pivot}$',
    'alpha_1': r'$\alpha_1$', 'alpha_2': r'$\alpha_2$', 'log10McutII': r'$\log_{10} M_{\mathrm{cut,II}}$',
    'log10FescIII': r'$\log_{10} f_{\mathrm{esc,III}}$', 'alpha_escIII': r'$\alpha_{\mathrm{esc,III}}$',
    'log10FescII': r'$\log_{10} f_{\mathrm{esc,II}}$', 'alpha_escII': r'$\alpha_{\mathrm{esc,II}}$',
    'log10Fx': r'$\log_{10} f_X$', 'Emin': r'$E_\mathrm{min}\,[\mathrm{keV}]$',
    'log_epsilon': r'$\log_{10} \epsilon$', 'alpha_s': r'$\alpha_s$', 'alpha_h': r'$\alpha_h$',
    'E_break': r'$E_\mathrm{break}\,[\mathrm{keV}]$',
}
# Axis limits = prior box, so an unconstrained parameter fills its panel
priors = NF.get_par_priors(row['run_mode'], saras=b('saras'), sfe_21cmFAST=b('sfe_21cmFAST'),
                           fixed=NF.parse_fixed(row['fixed']), popII_only=b('popII_only'))
gauss = {'alpha_s', 'alpha_h', 'E_break'}   # [mean, sigma], not bounds
ranges = [tuple(priors[n]) if n not in gauss else 0.999 for n in names]

color = '#2a6f97'
fig = corner.corner(
    pts, weights=np.exp(log_w - log_w.max()), labels=[LATEX.get(n, n) for n in names],
    range=ranges, bins=30, smooth=1.0, smooth1d=1.0, color=color,
    levels=(1 - np.exp(-0.5), 1 - np.exp(-2.0)),   # 1 and 2 sigma in 2D
    fill_contours=True, plot_datapoints=False, plot_density=False,
    contour_kwargs=dict(linewidths=1.2), hist_kwargs=dict(linewidth=1.5),
    quantiles=[0.16, 0.5, 0.84], show_titles=True, title_fmt='.2f',
    title_kwargs=dict(fontsize=11), label_kwargs=dict(fontsize=13),
    labelpad=0.05, max_n_ticks=4,
)
for ax in fig.axes:
    ax.tick_params(labelsize=9, direction='in')
mode = row['run_mode'].replace('_', r'\_')
fig.suptitle(f"nautilus chain {chain_num}: {mode}, "
             f"$n_\\mathrm{{eff}}={s.n_eff:.0f}$", fontsize=14, y=1.02)
plt.show()